# Walmart M5 Data Profiling

## Business Objective

Before building the ETL and forecasting pipeline, the source data must be inspected to understand its **structure, quality, scale, and relationships**.

This notebook profiles the Walmart M5 source files and any extracted external datasets used by the project. The goal is to identify the transformations and data-quality rules required before loading data into PostgreSQL.

This notebook focuses on:

- Dataset dimensions
- Column names and structure
- Data types
- Memory usage
- Missing values
- Duplicate records
- Identifier cardinality
- Date coverage
- Basic validity checks
- Dataset relationships and join keys
- ETL transformation requirements

> This notebook is for **data profiling**, not deep exploratory analysis. Trends, seasonality, demand behavior, pricing relationships, weather effects, and other analytical questions belong in the next EDA notebook.


## 1. Import Libraries

The following libraries are used to load and inspect the source datasets.


In [1]:
from pathlib import Path

import pandas as pd


## 2. Define Project Paths

The raw M5 files are stored in the project's `data/raw` directory.

External API extracts can be added to the same profiling workflow after they are created.


In [2]:
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"

calendar_path = RAW_DIR / "calendar.csv"
prices_path = RAW_DIR / "sell_prices.csv"
sales_validation_path = RAW_DIR / "sales_train_validation.csv"
sales_evaluation_path = RAW_DIR / "sales_train_evaluation.csv"
submission_path = RAW_DIR / "sample_submission.csv"


## 3. Load Source Datasets

Each source file is loaded into a pandas DataFrame so its structure and quality can be inspected before ETL transformations are finalized.


In [3]:
calendar = pd.read_csv(calendar_path)
prices = pd.read_csv(prices_path)
sales = pd.read_csv(sales_validation_path)
evaluation = pd.read_csv(sales_evaluation_path)
submission = pd.read_csv(submission_path)

datasets = {
    "calendar": calendar,
    "prices": prices,
    "sales_validation": sales,
    "sales_evaluation": evaluation,
    "submission": submission,
}


## 4. Dataset Inventory

The first profiling step is to confirm the size of each dataset.

Row and column counts help identify which files are small supporting tables and which require more memory-efficient processing.


In [4]:
for name, df in datasets.items():
    print(
        f"{name.upper():18} "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]:,}"
    )


CALENDAR           Rows: 1,969 | Columns: 14
PRICES             Rows: 6,841,121 | Columns: 4
SALES_VALIDATION   Rows: 30,490 | Columns: 1,919
SALES_EVALUATION   Rows: 30,490 | Columns: 1,947
SUBMISSION         Rows: 60,980 | Columns: 29


### Profiling Note

The M5 sales files are stored in a **wide format**, with one column per historical day. This structure is useful for the original competition format but is not appropriate for a relational fact table.

That structural issue will later require an ETL transformation from wide format to long format.


## 5. Preview Source Data

Previewing a few records helps confirm what each dataset contains and how the columns are organized.


### Calendar Dataset


In [5]:
calendar.head()


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


### Sell Prices Dataset


In [6]:
prices.head()


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


### Sales Validation Dataset


In [7]:
sales.head()


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


### Sales Evaluation Dataset


In [8]:
evaluation.head()


,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


### Sample Submission Dataset


In [9]:
submission.head()


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 6. Column and Data-Type Inspection

Data types are checked to identify dates stored as text, numeric fields, categorical identifiers, and any columns that may require type conversion during ETL.


In [10]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.dtypes)



CALENDAR
date              str
wm_yr_wk        int64
weekday           str
wday            int64
month           int64
year            int64
d                 str
event_name_1      str
event_type_1      str
event_name_2      str
event_type_2      str
snap_CA         int64
snap_TX         int64
snap_WI         int64
dtype: object

PRICES
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object

SALES_VALIDATION
id            str
item_id       str
dept_id       str
cat_id        str
store_id      str
            ...  
d_1909      int64
d_1910      int64
d_1911      int64
d_1912      int64
d_1913      int64
Length: 1919, dtype: object

SALES_EVALUATION
id            str
item_id       str
dept_id       str
cat_id        str
store_id      str
            ...  
d_1937      int64
d_1938      int64
d_1939      int64
d_1940      int64
d_1941      int64
Length: 1947, dtype: object

SUBMISSION
id       str
F1     int64
F2     int64
F3     int64
F4    

## 7. Memory Usage

Memory usage is important because the M5 dataset contains millions of records and very wide sales tables.

`deep=True` includes the memory used by object/string values, giving a more realistic estimate of RAM usage.


In [11]:
for name, df in datasets.items():
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2

    print(
        f"{name.upper():18} "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]:,} | "
        f"Memory: {memory_mb:,.2f} MB"
    )


CALENDAR           Rows: 1,969 | Columns: 14 | Memory: 0.26 MB
PRICES             Rows: 6,841,121 | Columns: 4 | Memory: 318.15 MB
SALES_VALIDATION   Rows: 30,490 | Columns: 1,919 | Memory: 448.23 MB
SALES_EVALUATION   Rows: 30,490 | Columns: 1,947 | Memory: 454.74 MB
SUBMISSION         Rows: 60,980 | Columns: 29 | Memory: 15.16 MB


## 8. Missing Value Analysis

Missing values are inspected before transformation so that they are not removed blindly.

A missing value may represent either:

- a genuine data-quality problem, or
- a valid business meaning, such as **no event occurring on a calendar date**.


In [12]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name.upper()}")

    if missing.empty:
        print("No missing values.")
    else:
        print(missing)



CALENDAR
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

PRICES
No missing values.

SALES_VALIDATION
No missing values.

SALES_EVALUATION
No missing values.

SUBMISSION
No missing values.


### Missing-Value Interpretation

The calendar event columns are expected to contain missing values because most dates do not correspond to holidays or special events.

These values should **not** be removed simply because they are null. Their meaning should be preserved during transformation and later encoded appropriately for modeling.


## 9. Duplicate Record Analysis

Exact duplicate rows are checked across all source files.

This determines whether basic duplicate-removal logic is necessary in the transformation layer.


In [13]:
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name.upper():18} Exact duplicate rows: {duplicate_count:,}")


CALENDAR           Exact duplicate rows: 0
PRICES             Exact duplicate rows: 0
SALES_VALIDATION   Exact duplicate rows: 0
SALES_EVALUATION   Exact duplicate rows: 0
SUBMISSION         Exact duplicate rows: 0


## 10. Business-Key Duplicate Checks

Exact duplicate rows are only one kind of duplication.

For warehouse tables, it is also important to check whether columns that should uniquely identify a record appear more than once.


In [14]:
# One price is expected for each store-item-week combination.
price_key_duplicates = prices.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

# Calendar day identifiers should be unique.
calendar_day_duplicates = calendar.duplicated(
    subset=["d"]
).sum()

# Calendar dates should be unique.
calendar_date_duplicates = calendar.duplicated(
    subset=["date"]
).sum()

print("Price store-item-week duplicates:", f"{price_key_duplicates:,}")
print("Calendar d duplicates:", f"{calendar_day_duplicates:,}")
print("Calendar date duplicates:", f"{calendar_date_duplicates:,}")


Price store-item-week duplicates: 0
Calendar d duplicates: 0
Calendar date duplicates: 0


## 11. Identifier Cardinality

Unique identifier counts help define the major dimensions in the future PostgreSQL warehouse.


In [15]:
identifier_counts = {
    "Stores": sales["store_id"].nunique(),
    "Products": sales["item_id"].nunique(),
    "Departments": sales["dept_id"].nunique(),
    "Categories": sales["cat_id"].nunique(),
    "States": sales["state_id"].nunique(),
}

for name, count in identifier_counts.items():
    print(f"{name}: {count:,}")


Stores: 10
Products: 3,049
Departments: 7
Categories: 3
States: 3


## 12. Date Coverage

Date coverage is checked so the ETL pipeline can confirm that the calendar spans the historical sales period and provides the required date mapping.


In [16]:
calendar_dates = pd.to_datetime(calendar["date"], errors="coerce")

print("Calendar start date:", calendar_dates.min())
print("Calendar end date:  ", calendar_dates.max())
print("Invalid calendar dates:", calendar_dates.isna().sum())

validation_day_columns = [col for col in sales.columns if col.startswith("d_")]
evaluation_day_columns = [col for col in evaluation.columns if col.startswith("d_")]

print("Validation sales day columns:", len(validation_day_columns))
print("Evaluation sales day columns:", len(evaluation_day_columns))
print("First validation day:", validation_day_columns[0])
print("Last validation day: ", validation_day_columns[-1])
print("First evaluation day:", evaluation_day_columns[0])
print("Last evaluation day: ", evaluation_day_columns[-1])


Calendar start date: 2011-01-29 00:00:00
Calendar end date:   2016-06-19 00:00:00
Invalid calendar dates: 0
Validation sales day columns: 1913
Evaluation sales day columns: 1941
First validation day: d_1
Last validation day:  d_1913
First evaluation day: d_1
Last evaluation day:  d_1941


## 13. Basic Value Validity Checks

These checks look for clearly invalid values that would require transformation or investigation.

They are intentionally simple and business-focused rather than exploratory.


In [17]:
print("Missing sell prices:", prices["sell_price"].isna().sum())
print("Zero or negative sell prices:", (prices["sell_price"] <= 0).sum())

sales_values = sales[validation_day_columns]

print("Missing sales values:", sales_values.isna().sum().sum())
print("Negative sales values:", (sales_values < 0).sum().sum())


Missing sell prices: 0
Zero or negative sell prices: 0
Missing sales values: 0
Negative sales values: 0


## 14. Dataset Relationships

The M5 source files connect through several shared identifiers:

- `item_id` identifies individual products.
- `store_id` identifies individual stores.
- `wm_yr_wk` connects weekly sell prices to the calendar.
- `d` connects daily sales columns such as `d_1`, `d_2`, ... to actual calendar dates.
- `dept_id` and `cat_id` define the product hierarchy.
- `state_id` connects stores to their states.

These relationships determine how the raw files will be transformed and joined when building the PostgreSQL warehouse.


## 15. ETL Transformation Requirements

Based on the profiling results, the ETL layer should implement only transformations that are justified by the source data.

### Calendar

- Convert `date` from text to a proper datetime value.
- Preserve missing event fields because they represent dates without special events.
- Retain `d` and `wm_yr_wk` because they are required to connect sales and prices to calendar dates.

### Sell Prices

- Retain `store_id`, `item_id`, `wm_yr_wk`, and `sell_price`.
- Ensure `sell_price` remains numeric.
- Enforce the expected store-item-week business key.
- Do not add duplicate or missing-value removal unless profiling or validation identifies a real issue.

### Sales

- Preserve the product/store hierarchy identifiers.
- Convert the wide `d_1`, `d_2`, ... daily sales columns into long format.
- Join the `d` identifier to the calendar to obtain actual dates.
- Keep sales values numeric and non-negative.

### Data Quality

- Do not remove expected calendar-event nulls.
- Do not add generic `dropna()` or `drop_duplicates()` logic unless a specific data problem justifies it.
- Use validation scripts after transformation and loading to confirm that warehouse-ready data satisfies the expected rules.


## 16. Profiling Findings

The initial source-data profiling shows that:

- The M5 data is split across related calendar, pricing, sales, evaluation, and submission files.
- The sales files are extremely wide and require a wide-to-long transformation before loading into the warehouse.
- The source files connect through identifiers such as `d`, `wm_yr_wk`, `item_id`, and `store_id`.
- Calendar event nulls are expected and should not be treated as corrupted records.
- Exact duplicate and business-key checks determine whether duplicate-handling logic is necessary.
- The profiling results define the concrete rules used by the transformation and validation layers.


## 17. Next Steps

With the source data profiled and the required transformations identified, the next stages are:

1. Update or implement the required `transform_*.py` logic.
2. Load the transformed datasets into PostgreSQL.
3. Run `validate_*.py` checks against the transformed/loaded data.
4. Perform deeper exploratory data analysis on the analysis-ready data.
5. Engineer forecasting features such as lagged sales, rolling averages, price changes, holidays, weather, and economic indicators.
6. Train and evaluate demand forecasting models.
7. Use the forecasts to support the dynamic pricing system.
